# 14 — Subtype classical ML (SVM / RF / KNN)

Same `train_classical_cv` pipeline as notebook 09. Per-fold JSON files saved with the `subtype_` prefix so they don't collide with the original lung-binary results.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import yaml

from utils.seed import set_seed
from utils.models import CLASSICAL_REGISTRY
from utils.training import train_classical_cv, save_fold_results
from utils.metrics import aggregate_folds
from utils.stats import format_results_table

with open('../configs/histology_subtype.yaml') as f:
    cfg = yaml.safe_load(f)
set_seed(cfg['seed'])

cache_dir = Path('..') / cfg['data']['cache_dir']
data = np.load(cache_dir / 'features.npz', allow_pickle=True)
X, y = data['X'], data['y']
print('X', X.shape, 'pos rate (aca):', y.mean())

In [ ]:
results_dir = Path('..') / cfg['paths']['results']
results_dir.mkdir(parents=True, exist_ok=True)

aggregated = {}
for name, (builder, grid) in CLASSICAL_REGISTRY.items():
    print(f'\n=== {name} ===')
    folds = train_classical_cv(
        X, y, builder, grid,
        n_splits=cfg['cv']['n_splits'],
        inner_splits=cfg['cv']['inner_splits'],
        scoring=cfg['cv']['scoring'],
        seed=cfg['seed'],
        verbose=True,
    )
    save_fold_results(folds, results_dir / f'subtype_{name}.json')
    aggregated[name] = aggregate_folds([f.metrics_calibrated for f in folds])

format_results_table(aggregated)